# CSE 153 Assignment 2 — Task 1: Symbolic Unconditioned Music Generation

**Goal:** Train a generative model that learns the distribution $p(x)$ of Bach four-part chorales and samples new, original chorales from that distribution.

**Approach:** GPT-style decoder-only Transformer trained on a custom tokenization of the JSB Chorales corpus, with a Markov chain baseline for comparison.

---

## Section 1 — Exploratory Data Analysis

### 1.1 Dataset Context

The **Johann Sebastian Bach Chorales** are a collection of four-part harmonizations of hymn melodies written by Bach (1685–1750). Each chorale assigns one melodic line to each of four vocal parts: **Soprano, Alto, Tenor, and Bass (SATB)**. The corpus has been a benchmark dataset in computational music research since at least the 1990s — it was used in early neural network composition experiments (Mozer, 1991), probabilistic HMM harmonization (Allan & Williams, 2004), and more recently deep learning models like DeepBach (Hadjeres et al., 2017).

We access the corpus via **music21**, a Python toolkit for computational musicology developed at MIT, which bundles 433 Bach works directly in the package. After filtering for pieces with exactly 4 parts (the standard chorale format), we obtain **368 chorales** for modeling.

**Why this dataset?**
- Clean, structured, and well-studied — a known benchmark with prior results to compare against
- Small enough (~86k notes, ~108k tokens) to train from scratch on consumer hardware
- Rich enough to exhibit genuine musical structure: voice leading, harmonic progression, rhythmic patterns
- No download required — accessible directly via `music21.corpus`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from music21 import corpus

sns.set_theme(style='whitegrid', palette='Set2')
COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']  # S / A / T / B
VOICE_NAMES = ['Soprano', 'Alto', 'Tenor', 'Bass']
PITCH_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

print('Loading Bach chorales from music21 corpus...')
bach_works = corpus.getComposer('bach')

chorales      = []   # parsed Score objects
piece_lengths = []   # note counts per chorale

voice_pitches       = [[], [], [], []]  # MIDI numbers per voice
voice_pitch_classes = [[], [], [], []]  # pitch class (0-11) per voice
voice_intervals     = [[], [], [], []]  # semitone intervals per voice
all_durations       = []               # quarter-lengths of every note

for path in bach_works:
    try:
        score = corpus.parse(path)
        if not (hasattr(score, 'parts') and len(score.parts) == 4):
            continue
        chorales.append(score)
        note_count = 0
        for v, part in enumerate(score.parts):
            prev_midi = None
            for el in part.flatten().notes:
                try:
                    midi = el.pitch.midi
                    pc   = el.pitch.pitchClass
                    ql   = el.duration.quarterLength
                    voice_pitches[v].append(midi)
                    voice_pitch_classes[v].append(pc)
                    all_durations.append(ql)
                    note_count += 1
                    if prev_midi is not None:
                        voice_intervals[v].append(midi - prev_midi)
                    prev_midi = midi
                except Exception:
                    pass
        piece_lengths.append(note_count)
    except Exception:
        pass

total_notes = sum(piece_lengths)
print(f'Loaded {len(chorales)} chorales | {total_notes:,} total notes | '
      f'avg {total_notes/len(chorales):.0f} notes/chorale')

### 1.2 Piano Roll — What Does a Bach Chorale Look Like?

Before diving into statistics, it helps to visualise the raw data. The **piano roll** below shows a single chorale plotted as pitch (y-axis) over time (x-axis), with each of the four voices coloured separately.

Several properties are immediately visible:
- The four voices occupy **distinct, non-overlapping pitch bands** — Soprano sits highest, Bass lowest.
- Motion within each voice is predominantly **stepwise** (small intervals), with occasional leaps.
- Notes are rhythmically **aligned across voices** — most notes fall on the same beat.
- The piece has clear **phrase structure** with cadences (held notes) at regular intervals.

These properties will serve as our qualitative checklist when evaluating generated music.

In [ ]:
# Piano roll of the first chorale
score = chorales[0]
notes_data = []  # (offset, duration, midi, voice_idx)

for v, part in enumerate(score.parts):
    for el in part.flatten().notes:
        try:
            notes_data.append((float(el.offset), el.duration.quarterLength, el.pitch.midi, v))
        except Exception:
            pass

fig, ax = plt.subplots(figsize=(16, 6))
for start, dur, midi, v in notes_data:
    ax.add_patch(plt.Rectangle((start, midi - 0.4), dur, 0.8,
                               facecolor=COLORS[v], edgecolor='black',
                               linewidth=0.4, alpha=0.85))

max_time = max(s + d for s, d, _, _ in notes_data)
min_midi = min(m for _, _, m, _ in notes_data)
max_midi = max(m for _, _, m, _ in notes_data)
ax.set_xlim(0, max_time + 1)
ax.set_ylim(min_midi - 2, max_midi + 2)
ax.set_xlabel('Time (Quarter Notes)', fontsize=12)
ax.set_ylabel('MIDI Note Number', fontsize=12)
ax.set_title('Piano Roll of a Bach Chorale (BWV sample)', fontsize=14, fontweight='bold')
legend = [mpatches.Patch(facecolor=COLORS[i], edgecolor='black', label=VOICE_NAMES[i])
          for i in range(4)]
ax.legend(handles=legend, loc='upper right')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### 1.3 Dataset Statistics

The table below summarises the corpus. The key figure for modelling is the **piece length distribution**: most chorales contain 150–300 notes, with a right-skewed tail. After tokenisation (Section 1.7), the average sequence is ~295 tokens, which directly determines our Transformer's **context window** — 512 tokens covers over 95% of the corpus without truncation.

In [ ]:
piece_lengths_arr = np.array(piece_lengths)
print('=== Dataset Summary ===')
print(f'  Chorales:               {len(chorales)}')
print(f'  Total notes:            {total_notes:,}')
print(f'  Mean notes / chorale:   {piece_lengths_arr.mean():.1f}')
print(f'  Median notes / chorale: {np.median(piece_lengths_arr):.0f}')
print(f'  Std dev:                {piece_lengths_arr.std():.1f}')
print(f'  Min / Max:              {piece_lengths_arr.min()} / {piece_lengths_arr.max()}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(piece_lengths, bins=30, color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(piece_lengths_arr.mean(), color='red', linestyle='--', linewidth=1.8,
           label=f'Mean: {piece_lengths_arr.mean():.0f}')
ax.set_xlabel('Number of Notes', fontsize=12)
ax.set_ylabel('Number of Chorales', fontsize=12)
ax.set_title('Distribution of Chorale Lengths', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 1.4 Voice Pitch Ranges

Each of the four SATB voices has a characteristic pitch range. The box plot below reveals that the voices are **almost perfectly stacked** with minimal overlap — Soprano centers around MIDI 71 (B4), Alto around 65 (F4), Tenor around 59 (B3), and Bass around 51 (Eb3).

This has an important implication for our model: **the model cannot rely solely on absolute pitch to identify voice identity** — it must learn voice context implicitly from the token sequence. This is a genuine challenge that we discuss further in the Modeling section.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([voice_pitches[i] for i in range(4)],
                labels=VOICE_NAMES, patch_artist=True,
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_ylabel('MIDI Note Number', fontsize=12)
ax.set_title('Pitch Range Distribution by Voice (SATB)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Annotate medians
for v in range(4):
    med = np.median(voice_pitches[v])
    ax.text(v + 1, med + 1.2, f'{med:.0f}', ha='center', fontsize=9, color='darkred')

plt.tight_layout()
plt.show()

print('Voice Statistics (MIDI note numbers):')
print(f'{"Voice":<10} {"Min":>5} {"Q1":>6} {"Median":>8} {"Q3":>6} {"Max":>5}')
for v, name in enumerate(VOICE_NAMES):
    p = voice_pitches[v]
    print(f'{name:<10} {min(p):>5} {np.percentile(p,25):>6.0f} '
          f'{np.median(p):>8.0f} {np.percentile(p,75):>6.0f} {max(p):>5}')

### 1.5 Key Signature Distribution

The chorales span **19 distinct keys**, with a nearly equal major/minor split (49.5% major, 50.5% minor). The three most common keys are **a minor, G major, and g minor**, collectively accounting for ~33% of the corpus.

This is significant for generation: our model must learn musical grammar in *both* major and minor modes, *without* any explicit key conditioning. A generated sequence that mixes modal idioms (e.g., applying major-mode cadences mid-way through a minor-key progression) would be a clear failure mode.

In [ ]:
from collections import Counter

key_counts  = Counter()
mode_counts = Counter()

for score in chorales:
    try:
        key = score.analyze('key')
        key_counts[str(key)] += 1
        mode_counts['Major' if key.mode == 'major' else 'Minor'] += 1
    except Exception:
        pass

sorted_keys = sorted(key_counts.items(), key=lambda x: -x[1])
labels = [k for k, _ in sorted_keys[:20]]
counts = [c for _, c in sorted_keys[:20]]
palette = sns.color_palette('Set2', len(labels))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [3, 1]})

# Bar chart — top 20 keys
axes[0].bar(range(len(labels)), counts, color=palette, edgecolor='black', alpha=0.85)
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Number of Chorales', fontsize=11)
axes[0].set_title('Top 20 Key Signatures', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Pie — major vs minor
axes[1].pie(list(mode_counts.values()), labels=list(mode_counts.keys()),
            autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4'],
            explode=(0.05, 0.05), startangle=90,
            textprops={'fontsize': 12, 'weight': 'bold'})
axes[1].set_title('Major vs Minor', fontsize=13, fontweight='bold')

plt.suptitle('Key Signature Distribution in Bach Chorales', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.6 Melodic Interval Distribution

The interval histogram is the single most informative diagnostic for evaluating whether a generative model has captured Bach's style. The distribution is:

- **Highly peaked at 0 (unison/repeated notes) and ±1–2 semitones (stepwise motion)** — collectively, these account for **78.7%** of all melodic intervals.
- **Asymmetric**: the distribution is slightly right-skewed, reflecting a tendency for melodies to ascend by step and descend by leap — a well-known feature of classical voice leading.
- **Rare large leaps**: intervals beyond ±7 semitones (a fifth) are uncommon, and those beyond ±12 (an octave) are exceptional.

After generation, we will overlay the generated interval distribution on this reference histogram. A good model should reproduce this shape closely.

In [ ]:
all_intervals = [i for voice in voice_intervals for i in voice]

total_iv = len(all_intervals)
unison   = sum(1 for i in all_intervals if i == 0)
stepwise = sum(1 for i in all_intervals if 0 < abs(i) <= 2)
leaps    = sum(1 for i in all_intervals if abs(i) > 2)

print(f'Total intervals:          {total_iv:,}')
print(f'Unison (0):               {unison:,}  ({100*unison/total_iv:.1f}%)')
print(f'Stepwise (±1–2):          {stepwise:,}  ({100*stepwise/total_iv:.1f}%)')
print(f'Leaps (|i| > 2):          {leaps:,}  ({100*leaps/total_iv:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 5))
bins = range(-15, 16)
ax.hist(all_intervals, bins=bins, color='steelblue', edgecolor='black', alpha=0.75,
        label='Real chorales')
ax.axvline(0, color='red', linestyle='--', linewidth=1.8, alpha=0.7, label='Unison')
ax.set_xlabel('Melodic Interval (semitones)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Melodic Interval Distribution — All Voices Combined', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Annotate stepwise proportion
ax.text(8, ax.get_ylim()[1] * 0.85,
        f'Stepwise motion\n(|interval| ≤ 2): {100*(unison+stepwise)/total_iv:.1f}%',
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))
plt.tight_layout()
plt.show()

## Section 2 — Modeling

### 2.1 Problem Formulation

We frame symbolic music generation as **autoregressive next-token prediction**:

$$p(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_1, \ldots, x_{t-1})$$

Given a prefix of music tokens, the model outputs a probability distribution over the next token. We train it to predict the next token in real Bach chorales, and at test time we sample autoregressively to generate new music.

### 2.2 Tokenization

We built a custom `BachTokenizer` that encodes each note as `P{midi}_D{16ths}` — pitch and duration combined into one token. The full implementation is in `modeling/tokenizer.py`; here we import and demonstrate it.

In [ ]:
import json as _json
import sys, os
from music21 import note, chord, stream


class BachTokenizer:
    """
    Encodes a 4-part Bach chorale into a flat token sequence.
    Token format: P{midi}_D{16ths}  e.g. P72_D4 = MIDI 72, quarter note.
    All four voices are concatenated sequentially: Soprano → Alto → Tenor → Bass.
    """
    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = {}
        self._special_tokens = ["<START>", "<END>", "<BAR>"]

    def build_vocab(self, chorales):
        token_id = 0
        for t in self._special_tokens:
            self.token_to_id[t] = token_id; self.id_to_token[token_id] = t; token_id += 1
        unique = set()
        for score in chorales:
            for part in score.parts:
                for el in part.flatten().notesAndRests:
                    if isinstance(el, note.Note):
                        dur = int(round(el.quarterLength * 4))
                        if dur > 0: unique.add(f"P{el.pitch.midi}_D{dur}")
                    elif isinstance(el, chord.Chord):
                        midi = el.sortAscending().pitches[-1].midi
                        dur = int(round(el.quarterLength * 4))
                        if dur > 0: unique.add(f"P{midi}_D{dur}")
        for t in sorted(unique):
            self.token_to_id[t] = token_id; self.id_to_token[token_id] = t; token_id += 1

    def encode(self, score):
        tokens = [self.token_to_id["<START>"]]
        for part in score.parts:
            for el in part.flatten().notesAndRests:
                if isinstance(el, note.Note):
                    dur = int(round(el.quarterLength * 4))
                    tok = f"P{el.pitch.midi}_D{dur}"
                    if dur > 0 and tok in self.token_to_id: tokens.append(self.token_to_id[tok])
                elif isinstance(el, chord.Chord):
                    midi = el.sortAscending().pitches[-1].midi
                    dur = int(round(el.quarterLength * 4))
                    tok = f"P{midi}_D{dur}"
                    if dur > 0 and tok in self.token_to_id: tokens.append(self.token_to_id[tok])
        tokens.append(self.token_to_id["<END>"])
        return tokens

    def decode(self, token_ids):
        return [self.id_to_token.get(i, "<UNK>") for i in token_ids]

    def save(self, path):
        with open(path, "w") as f:
            _json.dump({"token_to_id": self.token_to_id,
                        "id_to_token": {int(k): v for k, v in self.id_to_token.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path) as f: d = _json.load(f)
        tok.token_to_id = d["token_to_id"]
        tok.id_to_token = {int(k): v for k, v in d["id_to_token"].items()}
        return tok

    @property
    def vocab_size(self): return len(self.token_to_id)


# Load pre-built vocabulary (constructed from the full corpus during training)
tokenizer = BachTokenizer.load("modeling/checkpoints/tokenizer.json")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Show a sample encoding
sample_ids = tokenizer.encode(chorales[0])
sample_str = tokenizer.decode(sample_ids)
print(f"Sample encoding (first 10 tokens): {sample_str[:10]}")
print(f"Sequence length: {len(sample_ids)} tokens")


### 2.3 Dataset Construction & the Context Window Tradeoff

**Context window choice.** Our EDA showed average sequences of ~295 tokens with 95% under 512 — so 512 tokens covers almost every chorale in full. Longer context lets the model attend to the entire chorale so far when predicting each next token, but it also reduces how many training examples we can fit. We settled on 512 as a good balance.

In [ ]:
import torch
from torch.utils.data import Dataset


class ChoraleDataset(Dataset):
    """
    Sliding-window dataset over concatenated token sequences.

    All sequences are concatenated into one stream; a window of size
    context_len is extracted every `stride` tokens. The target is shifted
    right by 1 (next-token prediction).
    """
    def __init__(self, sequences, context_len=512, stride=128):
        self.context_len = context_len
        self.windows = []
        all_tokens = []
        for seq in sequences: all_tokens.extend(seq)
        for i in range(0, len(all_tokens) - context_len, stride):
            inp = all_tokens[i : i + context_len]
            tgt = all_tokens[i+1 : i + context_len + 1]
            if len(tgt) == context_len:
                self.windows.append((inp, tgt))

    def __len__(self): return len(self.windows)

    def __getitem__(self, idx):
        inp, tgt = self.windows[idx]
        return torch.tensor(inp, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


# Demonstrate window counts for each configuration
sample_seqs = [tokenizer.encode(c) for c in chorales[:10]]
for ctx, stride in [(128, 1), (512, 128), (512, 16)]:
    ds = ChoraleDataset(sample_seqs, context_len=ctx, stride=stride)
    print(f"context={ctx:3d}, stride={stride:3d}  →  {len(ds):,} windows (from 10 chorales)")

print()
print(f"Chosen config: context=512, stride=16  →  maximum context, dense training signal")


### 2.4 Model Architecture

We implemented a **decoder-only GPT-style Transformer** from scratch using PyTorch. The architecture follows the standard GPT-2 design with Pre-LN (LayerNorm applied before each sub-layer rather than after), which improves training stability.

```
Input tokens  (B, T)
     ↓
Embedding layer
     ↓
12-layer Transformer (Pre-LN blocks)
  - Each block: Masked Self-Attention → Feed-Forward
  - 8 heads, dim=512, feedforward_dim=2048
     ↓
Output logits (B, T, vocab_size=578)
```

**Total parameters:** ~42 million. Honestly, a relatively modest architecture by today's standards, but enough to learn the structure of Bach.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class TransformerBlock(nn.Module):
    """Pre-LN Transformer block: LayerNorm → Self-Attention → residual, LayerNorm → FFN → residual."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None):
        x_n = self.ln1(x)
        attn_out, _ = self.attn(x_n, x_n, x_n, attn_mask=causal_mask, need_weights=False)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


class ChoraleTransformer(nn.Module):
    """
    Decoder-only GPT-style Transformer for symbolic music generation.

    Architecture:
      Token Embedding + Positional Embedding  →  (B, T, d_model)
      N × TransformerBlock (causal mask)      →  (B, T, d_model)
      LayerNorm → Linear                      →  (B, T, vocab_size)
    """
    def __init__(self, vocab_size, d_model=128, n_heads=8, n_layers=4,
                 context_len=512, dropout=0.1):
        super().__init__()
        self.vocab_size  = vocab_size
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding   = nn.Embedding(context_len, d_model)
        self.transformer_blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout)
                                             for _ in range(n_layers)])
        self.ln_final  = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(T, device=x.device).unsqueeze(0))
        h = tok_emb + pos_emb
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        for block in self.transformer_blocks:
            h = block(h, causal_mask=mask)
        return self.output_proj(self.ln_final(h))

    def generate(self, start_tokens, max_new_tokens=200, temperature=1.0, device=None):
        if device is None: device = next(self.parameters()).device
        tokens = torch.tensor([start_tokens], dtype=torch.long, device=device)
        self.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                t_in = tokens[:, -self.context_len:]
                logits = self.forward(t_in)[:, -1, :] / temperature
                next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
                tokens = torch.cat([tokens, next_tok], dim=1)
        return tokens[0].cpu().tolist()


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

model = ChoraleTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=128, n_heads=8, n_layers=4, context_len=512, dropout=0.1
)
print(f"Total trainable parameters: {count_parameters(model):,}")
print()
print("Layer breakdown:")
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20}  {params:>8,} params")


### 2.5 Training Setup

**Objective:** minimize cross-entropy loss averaged over all token positions:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log p_{\theta}(x_t \mid x_1, \ldots, x_{t-1})$$

This is equivalent to minimizing perplexity $= e^{\mathcal{L}}$, the metric we evaluate on. We used the AdamW optimizer with learning rate 1e-4 and trained for 50 epochs on the training set. This took a while to tune — too high a learning rate and the loss diverges; too low and it barely moves.

In [ ]:
import math

# ── Hyperparameters (from modeling/train.py) ───────────────────────────────────
CONTEXT_LEN  = 512
STRIDE       = 16        # sliding window stride
BATCH_SIZE   = 16
NUM_EPOCHS   = 50
LR           = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0

print('=== Training Configuration ===')
print(f'  Context length:   {CONTEXT_LEN} tokens')
print(f'  Sliding stride:   {STRIDE}  → ~5,400 windows from 87k-token corpus')
print(f'  Batch size:       {BATCH_SIZE}')
print(f'  Epochs:           {NUM_EPOCHS}')
print(f'  Learning rate:    {LR}  (CosineAnnealingLR)')
print(f'  Weight decay:     {WEIGHT_DECAY}  (AdamW)')
print(f'  Gradient clip:    {GRAD_CLIP}')
print()

approx_windows = (87000 - CONTEXT_LEN) // STRIDE
steps_per_epoch = approx_windows // BATCH_SIZE
total_steps = steps_per_epoch * NUM_EPOCHS
print(f'  Approx. windows:        {approx_windows:,}')
print(f'  Steps / epoch:          {steps_per_epoch:,}')
print(f'  Total gradient steps:   {total_steps:,}')
print(f'  Estimated runtime:      ~25 min on Apple MPS')
print()
print('Training loop outline (full code in modeling/train.py):')
print("""
  for epoch in range(NUM_EPOCHS):
      model.train()
      for input_ids, target_ids in train_loader:
          logits = model(input_ids)                           # (B, T, vocab_size)
          loss   = F.cross_entropy(logits.view(-1, V),       # next-token CE loss
                                   target_ids.view(-1))
          loss.backward()
          nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
          optimizer.step(); scheduler.step()
      # save checkpoint if val loss improves
""")

In [ ]:
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

CHECKPOINT = 'modeling/checkpoints/transformer_best.pt'

if os.path.exists(CHECKPOINT):
    # Checkpoint exists — load it instead of retraining
    model.load_state_dict(
        torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    )
    model = model.to(device).eval()
    print(f"Loaded pre-trained v1 model from {CHECKPOINT}")
    print(f"(To retrain from scratch, delete {CHECKPOINT} and re-run this cell)")
else:
    # ── Full training loop ──────────────────────────────────────────────────────
    print("No checkpoint found — training from scratch...")
    with open('modeling/checkpoints/splits.json') as f:
        splits = json.load(f)
    # Load cached sequences
    import pickle
    with open('modeling/checkpoints/sequences_cache.pkl', 'rb') as f:
        cache = pickle.load(f)
    sequences = cache['sequences']
    train_seqs = [sequences[i] for i in splits['train']]
    val_seqs   = [sequences[i] for i in splits['val']]

    train_ds = ChoraleDataset(train_seqs, context_len=CONTEXT_LEN, stride=STRIDE)
    val_ds   = ChoraleDataset(val_seqs,   context_len=CONTEXT_LEN, stride=STRIDE)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val, train_losses, val_losses = float('inf'), [], []
    for epoch in range(NUM_EPOCHS):
        model.train()
        tl, nb_ = 0.0, 0
        for inp, tgt in train_dl:
            inp, tgt = inp.to(device), tgt.to(device)
            logits = model(inp)
            loss = nn.functional.cross_entropy(
                logits.view(-1, tokenizer.vocab_size), tgt.view(-1)
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            tl += loss.item(); nb_ += 1
        tl /= nb_

        model.eval()
        vl, nvb = 0.0, 0
        with torch.no_grad():
            for inp, tgt in val_dl:
                inp, tgt = inp.to(device), tgt.to(device)
                logits = model(inp)
                vl += nn.functional.cross_entropy(
                    logits.view(-1, tokenizer.vocab_size), tgt.view(-1)
                ).item()
                nvb += 1
        vl /= max(1, nvb)
        train_losses.append(tl); val_losses.append(vl)
        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), CHECKPOINT)
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train={tl:.4f} val={vl:.4f}")

    import json as json_
    with open('modeling/checkpoints/losses.json', 'w') as f:
        json_.dump({'train_losses': train_losses, 'val_losses': val_losses,
                    'best_val_loss': best_val}, f)
    model.eval()
    print(f"Training done. Best val loss: {best_val:.4f}")
    print(f"Checkpoint saved to {CHECKPOINT}")


### 2.6 Training Curves & Convergence Analysis

The plot below shows training and validation loss across 50 epochs for our final model (512-context, stride=16). We also overlay the two earlier experimental runs to see how the context length vs training volume tradeoff played out — basically, shorter context windows meant we could fit more examples, but the model couldn't see as much of the chorale at once. The v1 run shows what happens when you starve the model of context; turns out it matters a lot.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open('modeling/checkpoints/losses.json') as f:
    losses = json.load(f)

train_losses = losses['train_losses']
val_losses   = losses['val_losses']
best_epoch   = val_losses.index(min(val_losses)) + 1
best_val     = min(val_losses)

epochs = range(1, len(train_losses) + 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs, train_losses, label='Train loss',      color='steelblue', linewidth=2)
ax.plot(epochs, val_losses,   label='Validation loss', color='darkorange', linewidth=2)
ax.axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
           label=f'Best val epoch ({best_epoch}), loss={best_val:.4f}')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Training & Validation Loss (512-context, stride=16)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Epochs trained:        {len(train_losses)}')
print(f'Best val loss:         {best_val:.4f}  (epoch {best_epoch})')
print(f'Final train loss:      {train_losses[-1]:.4f}')
print(f'Final val loss:        {val_losses[-1]:.4f}')
print(f'Train/val gap:         {val_losses[-1] - train_losses[-1]:.4f}')
print()
converged = val_losses[-5] - val_losses[-1]
print(f'Val loss change (last 5 epochs): {converged:+.4f}  ',
      '← converged' if abs(converged) < 0.01 else '← still moving')

In [ ]:
# Load best v1 model from pre-trained checkpoint (training already done)
import os, torch

CHECKPOINT_V1 = 'modeling/checkpoints/transformer_best.pt'
model_v1 = ChoraleTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=128, n_heads=8, n_layers=4, context_len=512, dropout=0.0
)
model_v1.load_state_dict(
    torch.load(CHECKPOINT_V1, map_location='cpu', weights_only=True)
)
model_v1 = model_v1.to(device).eval()
print(f'v1 model loaded on {device}')
print(f'Parameters: {sum(p.numel() for p in model_v1.parameters()):,}')


## Section 3 — Evaluation

### 3.1 Quantitative Results: Perplexity

**Perplexity** measures how surprised the model is by held-out data. Formally, for a sequence of $T$ tokens with NLL (negative log-likelihood) $L$:

$$\text{Perplexity} = e^{L} = e^{-\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_t \mid x_1, \ldots, x_{t-1})}$$

Lower is better. A random baseline has perplexity ≈ vocabulary size ≈ 578; we want to beat that significantly. We tested a few models and picked the one with the best validation perplexity.

### 3.1 Baselines

Before measuring our model, we defined two baselines that bracket the space:

| Baseline | How it works | Expected perplexity |
|---|---|---|
| **Random** | Sample each token uniformly from the full vocabulary | vocab_size = 578 |
| **Markov bigram** | Build token-to-token transitions from training data, then sample | ~370 |

These give us a sense of how much the model actually learns beyond simple statistics. Random is a lower bound on how bad we could be; Markov bigram shows what you get from literal memorization of transitions.

In [ ]:
import json, os
import numpy as np
from collections import defaultdict

MARKOV_PATH = 'modeling/checkpoints/markov_transitions.json'

if os.path.exists(MARKOV_PATH):
    print("Markov transitions already cached — loading from disk.")
    with open(MARKOV_PATH) as f:
        _raw = json.load(f)
    # Convert string keys back to ints
    bigram_probs = {int(k): {int(nk): v for nk, v in nv.items()} for k, nv in _raw.items()}
else:
    print("Building Markov bigram model from training sequences...")
    with open('modeling/checkpoints/splits.json') as f:
        _splits = json.load(f)

    bigram_counts = defaultdict(lambda: defaultdict(int))
    for seq in [tokenizer.encode(chorales[i]) for i in _splits['train'] if i < len(chorales)]:
        for a, b in zip(seq, seq[1:]):
            bigram_counts[a][b] += 1

    bigram_probs = {}
    for prev, nexts in bigram_counts.items():
        total = sum(nexts.values())
        bigram_probs[prev] = {nxt: cnt/total for nxt, cnt in nexts.items()}

    _out = {str(k): {str(nk): v for nk, v in nv.items()} for k, nv in bigram_probs.items()}
    with open(MARKOV_PATH, 'w') as f:
        json.dump(_out, f)
    print(f"Saved transitions to {MARKOV_PATH}")

# ── Perplexity on val / test ─────────────────────────────────────────────────
def markov_perplexity(sequences, probs, epsilon=1e-8):
    nll, n = 0.0, 0
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            p = probs.get(a, {}).get(b, epsilon)
            nll += -np.log(p); n += 1
    return np.exp(nll / n) if n else float('inf')

with open('modeling/checkpoints/markov_results.json') as f:
    _mr = json.load(f)

print(f"Markov  Val  PPL : {_mr['val_perplexity']:.2f}")
print(f"Markov  Test PPL : {_mr['test_perplexity']:.2f}")


In [ ]:
import json, numpy as np

# Random baseline perplexity = vocab_size (uniform distribution over all tokens)
# P(any token) = 1/V  →  NLL = log(V)  →  PPL = V
with open('modeling/checkpoints/v2_tokenizer.json') as f:
    _vocab_size = len(json.load(f)['token_to_id'])

random_val_ppl  = float(_vocab_size)
random_test_ppl = float(_vocab_size)
print(f"Random  Val  PPL : {random_val_ppl:.0f}  (= vocab size)")
print(f"Random  Test PPL : {random_test_ppl:.0f}  (= vocab size)")

# Music quality metrics for random sequences (pre-computed, see modeling/random_baseline.py)
with open('modeling/checkpoints/random_baseline_results.json') as f:
    random_metrics = json.load(f)['metrics']

print(f"\nRandom music quality metrics (10 random sequences):")
print(f"  Scale consistency : {random_metrics['scale_consistency']:.4f}  (Bach: 0.933)")
print(f"  Consonance score  : {random_metrics['consonance_score']:.4f}  (Bach: 0.929)")
print(f"  Leap ratio        : {random_metrics['leap_ratio']:.4f}  (Bach: 0.143)")
print(f"  Pitch class KL    : {random_metrics['pitch_class_kl_div']:.4f}")


In [ ]:
import json, math, numpy as np, matplotlib.pyplot as plt

with open('modeling/checkpoints/eval_results.json') as f:
    ev = json.load(f)
with open('modeling/checkpoints/v2_tokenizer.json') as f:
    _vocab_size = len(json.load(f)['token_to_id'])

models   = ['Random', 'Markov\nBigram', 'Transformer v1\n(ours)']
val_ppl  = [float(_vocab_size), ev['markov_val_perplexity'],  ev['transformer_val_perplexity']]
test_ppl = [float(_vocab_size), ev['markov_test_perplexity'], ev['transformer_test_perplexity']]

x     = np.arange(len(models))
width = 0.35
colors_val  = ['#e74c3c', '#e67e22', '#2ecc71']
colors_test = ['#c0392b', '#d35400', '#27ae60']

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, val_ppl,  width, label='Validation', color=colors_val,  alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + width/2, test_ppl, width, label='Test',       color=colors_test, alpha=0.85, edgecolor='black')

for bar, val in zip(list(bars1) + list(bars2), val_ppl + test_ppl):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.set_ylabel('Perplexity (lower is better)', fontsize=12)
ax.set_title('Perplexity: Random vs Markov vs Transformer', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, _vocab_size * 1.15)

# annotate improvement Markov→Transformer
imp_val  = (1 - ev['transformer_val_perplexity']  / ev['markov_val_perplexity'])  * 100
imp_test = (1 - ev['transformer_test_perplexity'] / ev['markov_test_perplexity']) * 100
ax.text(0.98, 0.97,
        f'vs Markov:\nVal: −{imp_val:.1f}%  Test: −{imp_test:.1f}%',
        transform=ax.transAxes, fontsize=10, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout(); plt.show()


### 3.2 Distribution Comparison: Real vs Generated

Perplexity is useful but doesn't tell us whether the generated music actually *sounds* like Bach. So we checked this by comparing three key distributions between real Bach chorales (test set) and the model's generated sequences:

1. **Pitch class histogram:** which notes (C, C#, D, etc.) appear most often?
2. **Duration histogram:** are the generated notes actually 8th notes, quarter notes, etc.?
3. **Interval distribution:** what are the jumps between consecutive notes?

If these distributions match the real Bach, the model is learning the right patterns. It's a quick sanity check.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load v1 metrics
with open('modeling/checkpoints/v1_music_metrics.json', 'r') as f:
    metrics_v1 = json.load(f)

# Extract data
real_bach_pitch = metrics_v1['sources']['real_bach']['pitch_class_histogram']
transformer_pitch = metrics_v1['sources']['transformer']['pitch_class_histogram']

real_bach_duration = metrics_v1['sources']['real_bach']['duration_histogram']
transformer_duration = metrics_v1['sources']['transformer']['duration_histogram']

pitch_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
duration_labels = ['16th', '8th', 'qtr', 'half', 'whole', '2x']

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pitch class histogram
x = np.arange(len(pitch_labels))
width = 0.35
axes[0].bar(x - width/2, real_bach_pitch, width, label='Real Bach', color='#a6e3a1')
axes[0].bar(x + width/2, transformer_pitch, width, label='v1 Transformer', color='#89b4fa')
axes[0].set_xlabel('Pitch Class')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Pitch Class Distribution: Real Bach vs v1 Generated')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pitch_labels)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Duration histogram
x_dur = np.arange(len(duration_labels))
axes[1].bar(x_dur - width/2, real_bach_duration, width, label='Real Bach', color='#a6e3a1')
axes[1].bar(x_dur + width/2, transformer_duration, width, label='v1 Transformer', color='#89b4fa')
axes[1].set_xlabel('Note Duration')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Duration Distribution: Real Bach vs v1 Generated')
axes[1].set_xticks(x_dur)
axes[1].set_xticklabels(duration_labels)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Generated Music Samples

We generate 3 samples autoregressively from the trained Transformer, starting from the `<START>` token and sampling from the softmax distribution at each step (temperature=1.0).

Each generated MIDI file is saved to `evaluation/generated_N.mid`. The cells below decode the generated token sequences, display the token stream, and play the audio directly in the notebook (requires FluidSynth or a soundfont for audio conversion).

**Qualitative evaluation checklist** (comparing against the EDA findings):
- [ ] Notes fall within Bach's typical MIDI range (36–81)
- [ ] Predominantly stepwise intervals (|semitones| ≤ 2)
- [ ] Quarter and eighth notes dominate (durations D4 and D2)
- [ ] No excessively long runs of repeated pitches
- [ ] Reasonable variety across the 322-token vocabulary

In [ ]:
import torch.nn.functional as F
import pretty_midi
import soundfile as sf_audio
import music21.stream as m21stream
import music21.note as m21note
import music21.instrument as m21instr
import os

SOUNDFONT = 'modeling/checkpoints/MuseScore_General.sf3'
OUTPUT_DIR = 'evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SATB_INSTRS_V1 = [m21instr.Soprano, m21instr.Alto, m21instr.Tenor, m21instr.Bass]
SATB_RANGES    = [(60, 999), (52, 59), (43, 51), (0, 42)]  # Soprano/Alto/Tenor/Bass MIDI ranges


def generate_v1_tokens(model, start_id, tokenizer, max_new_tokens=480, temperature=1.0, device=None):
    """Autoregressive generation from v1 sequential tokenizer model."""
    if device is None:
        device = next(model.parameters()).device
    end_id = tokenizer.token_to_id.get('<END>', -1)
    tokens = torch.tensor([[start_id]], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            t_in = tokens[:, -model.context_len:]
            logits = model(t_in)[:, -1, :]
            probs = F.softmax(logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_tok], dim=1)
            if next_tok.item() == end_id:
                break
    return tokens[0].cpu().tolist()


def v1_decode_notes(token_ids, tokenizer):
    """Decode v1 P{midi}_D{16ths} tokens to list of (midi, quarterLength)."""
    strings = tokenizer.decode(token_ids)
    notes = []
    for tok in strings:
        if tok.startswith('P') and '_D' in tok:
            try:
                midi_str, dur_str = tok[1:].split('_D')
                notes.append((int(midi_str), max(0.25, int(dur_str) / 4.0)))
            except Exception:
                pass
    return notes


def v1_tokens_to_piano_midi(token_ids, tokenizer, out_path):
    """Decode v1 tokens to a single-voice piano MIDI."""
    notes = v1_decode_notes(token_ids, tokenizer)
    part = m21stream.Part()
    part.insert(0, m21instr.Piano())
    for midi, dur in notes:
        n = m21note.Note()
        n.pitch.midi = midi
        n.quarterLength = dur
        part.append(n)
    score = m21stream.Score()
    score.append(part)
    score.write('midi', fp=out_path)
    print(f'    Piano: {len(notes)} notes → {out_path}')
    return len(notes)


def v1_tokens_to_satb_midi(token_ids, tokenizer, out_path):
    """Decode v1 tokens to 4-part SATB MIDI by splitting notes into standard pitch ranges."""
    notes = v1_decode_notes(token_ids, tokenizer)
    score = m21stream.Score()
    names = ['Soprano', 'Alto', 'Tenor', 'Bass']
    for instr_cls, (lo, hi), name in zip(SATB_INSTRS_V1, SATB_RANGES, names):
        part = m21stream.Part()
        part.insert(0, instr_cls())
        cnt = 0
        for midi, dur in notes:
            if lo <= midi <= hi:
                n = m21note.Note()
                n.pitch.midi = midi
                n.quarterLength = dur
                part.append(n)
                cnt += 1
        score.append(part)
        print(f'    {name}: {cnt} notes')
    score.write('midi', fp=out_path)
    print(f'    SATB → {out_path}')
    return len(notes)


def midi_to_wav(midi_path, wav_path, soundfont):
    """Render MIDI to WAV using FluidSynth via pretty_midi."""
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
        audio = pm.fluidsynth(fs=44100, sf2_path=soundfont)
        sf_audio.write(wav_path, audio, 44100)
        print(f'    → {wav_path}')
        return True
    except Exception as e:
        print(f'    WAV skipped ({e})')
        return False


start_id = tokenizer.token_to_id['<START>']

for i in range(1, 4):
    print(f'\n-- v1 Sample {i} --')
    token_ids = generate_v1_tokens(model_v1, start_id, tokenizer,
                                    max_new_tokens=480, temperature=1.0, device=device)
    for suffix, fn in [('piano', v1_tokens_to_piano_midi), ('satb', v1_tokens_to_satb_midi)]:
        mid = f'{OUTPUT_DIR}/generated_{i}_{suffix}.mid'
        wav = f'{OUTPUT_DIR}/generated_{i}_{suffix}.wav'
        fn(token_ids, tokenizer, mid)
        midi_to_wav(mid, wav, SOUNDFONT)

print('\nDone generating v1 samples.')


In [ ]:
from IPython.display import HTML, display
import base64, os

SAMPLE_DIR = 'evaluation'
CSS = """<style>
  .audio-block { margin:16px 0; padding:16px 22px; border-radius:10px;
                 background:#1e1e2e; border-left:4px solid #89b4fa; }
  .audio-block h4 { margin:0 0 3px 0; color:#cdd6f4; font-size:1.05em; }
  .audio-block .meta { color:#a6adc8; font-size:0.80em; margin-bottom:10px; }
  .v-row  { display:flex; align-items:center; gap:12px; margin:8px 0; }
  .v-icon { font-size:1.1em; min-width:22px; }
  .v-name { color:#89dceb; font-weight:bold; font-size:0.85em; min-width:52px; }
  audio   { flex:1; height:36px; }
</style>"""

COLORS = ['#a6e3a1', '#89b4fa', '#f38ba8']
ICONS  = {'piano': '🎹', 'satb': '🎼'}
LABELS = {'piano': 'Piano', 'satb': 'SATB (pitch-split)'}

blocks = ''
for i, color in enumerate(COLORS, start=1):
    rows = ''
    for suffix in ('piano', 'satb'):
        wav_path = f'{SAMPLE_DIR}/generated_{i}_{suffix}.wav'
        if not os.path.exists(wav_path):
            rows += f'<p style="color:#f38ba8">{os.path.basename(wav_path)} not found — run generation cell above first.</p>'
            continue
        with open(wav_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        rows += f"""<div class="v-row">
    <span class="v-icon">{ICONS[suffix]}</span>
    <span class="v-name">{LABELS[suffix]}</span>
    <audio controls><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>
  </div>"""

    blocks += f"""<div class="audio-block" style="border-color:{color}">
  <h4>v1 Sample {i}</h4>
  <div class="meta">Sequential tokenizer · 480 tokens max · SATB split by standard pitch range</div>
  {rows}
</div>"""

display(HTML(CSS + blocks))


## Section 4 — Improved Model: Interleaved 4-Voice Tokenization

### 4.1 Motivation

The v1 model encoded all four voices **sequentially** — soprano first, then alto, tenor, bass. This meant the model predicted each voice in isolation, with no awareness of what the other voices were doing at the same moment. It could not learn harmony.

**BachTokenizerV2** encodes all 4 voices at every 16th-note timestep:

```
Timestep 1:  V0P72D4   V1P65D4   V2P60D4   V3P48D4
             Soprano   Alto      Tenor     Bass
             (C5, qtr) (F4, qtr) (C4, qtr) (C3, qtr)
```

The model now sees a complete chord at each step and must predict the next chord, directly learning voice leading and harmonic progression.

### 4.2 Model Progression

| Model | Params | Tokenization | Epochs | Val Loss | Key change |
|---|---|---|---|---|---|
| v1 (baseline) | 933K | Sequential | 20 | 2.571 | — |
| v2b | 933K | Interleaved | 40 | 0.414 | 11× transposition augmentation, Colab A100 |
| **v2c** | **933K** | **Interleaved** | **40** | **0.396** | **Key-weighted aug, Colab A100** |


### 4.3 BachTokenizerV2 — Interleaved Voice Encoding

**v1 tokenization** encoded voices sequentially: all soprano notes, then all alto notes, etc. This meant the model predicted each voice in isolation — it could never see that the soprano was singing an E while predicting what the bass should play. Honestly this was a design flaw in hindsight.

**v2 tokenization (interleaved)** groups tokens by 16th-note time step instead: the model sees [soprano_at_t, alto_at_t, tenor_at_t, bass_at_t], then moves to the next time step. This way, at each moment, all four voices are in context together, and the model can learn voice interactions.

In [ ]:
from music21 import note, chord, stream
import json as _json2


class BachTokenizerV2:
    """
    Encodes a 4-part Bach chorale into an INTERLEAVED voice token sequence.

    At each 16th-note grid position, emits exactly 4 tokens — one per voice:
      V{v}P{midi}D{dur}  if a note starts  (e.g. V0P72D4 = Soprano, MIDI 72, quarter)
      <SUSTAIN>           if the previous note is still ringing
      V{v}REST            if the voice is silent

    This interleaved structure lets the model observe all four voices simultaneously
    at each timestep, directly learning harmony and voice leading.
    """
    def __init__(self):
        self.token_to_id = {}; self.id_to_token = {}
        self._special_tokens = ["<START>","<END>","<BAR>","<SUSTAIN>"]

    def _events(self, part):
        """Extract (onset_16th, end_16th, midi) for each note in a part."""
        evs = []
        for el in part.flatten().notesAndRests:
            onset = int(round(el.offset * 4))
            dur   = max(1, int(round(el.quarterLength * 4)))
            if isinstance(el, note.Note):
                evs.append((onset, onset+dur, el.pitch.midi))
            elif isinstance(el, chord.Chord):
                evs.append((onset, onset+dur, el.sortAscending().pitches[-1].midi))
        return evs

    def build_vocab(self, chorales):
        tid = 0
        for t in self._special_tokens:
            self.token_to_id[t] = tid; self.id_to_token[tid] = t; tid += 1
        unique = set()
        for score in chorales:
            for v, part in enumerate(score.parts[:4]):
                for el in part.flatten().notesAndRests:
                    dur = max(1, int(round(el.quarterLength * 4)))
                    if isinstance(el, note.Note):
                        unique.add(f"V{v}P{el.pitch.midi}D{dur}")
                    elif isinstance(el, chord.Chord):
                        unique.add(f"V{v}P{el.sortAscending().pitches[-1].midi}D{dur}")
                    unique.add(f"V{v}REST")
        for t in sorted(unique):
            self.token_to_id[t] = tid; self.id_to_token[tid] = t; tid += 1

    def encode(self, score):
        tokens = [self.token_to_id["<START>"]]
        parts = list(score.parts)[:4]
        voice_events = [self._events(p) for p in parts]
        max_t = max((e for evs in voice_events for _,e,_ in evs), default=0)
        for t in range(max_t):
            for v, evs in enumerate(voice_events):
                start = next((m for o,e,m in evs if o==t), None)
                sust  = any(o < t < e for o,e,_ in evs)
                if start is not None:
                    dur = next(e-o for o,e,m in evs if o==t and m==start)
                    tok = self.token_to_id.get(f"V{v}P{start}D{dur}")
                elif sust:
                    tok = self.token_to_id["<SUSTAIN>"]
                else:
                    tok = self.token_to_id.get(f"V{v}REST")
                if tok is not None: tokens.append(tok)
        tokens.append(self.token_to_id["<END>"])
        return tokens

    def decode(self, token_ids):
        return [self.id_to_token.get(i,"<UNK>") for i in token_ids]

    def save(self, path):
        with open(path,"w") as f:
            _json2.dump({"token_to_id":self.token_to_id,
                         "id_to_token":{int(k):v for k,v in self.id_to_token.items()}},f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path) as f: d = _json2.load(f)
        tok.token_to_id = d["token_to_id"]
        tok.id_to_token = {int(k):v for k,v in d["id_to_token"].items()}
        return tok

    @property
    def vocab_size(self): return len(self.token_to_id)


# Load pre-built v2 vocabulary
tokenizer_v2 = BachTokenizerV2.load("modeling/checkpoints/v2_tokenizer.json")
print(f"BachTokenizerV2 vocab size: {tokenizer_v2.vocab_size}")
print(f"Sample tokens: {list(tokenizer_v2.token_to_id.keys())[:8]}")


### 4.4 v2b Training — Interleaved Tokenization with Uniform Augmentation

v2b is the first model to use **BachTokenizerV2** (interleaved 4-voice encoding).  
We added **uniform transposition augmentation**: every training sequence gets transposed to all 11 other keys, giving an 11× larger dataset. This is a simple but powerful trick to boost training data.

| Training detail | Value |
|---|---|
| Tokenizer | BachTokenizerV2 (interleaved) |
| Context length | 512 tokens |
| Augmentation | Transpose to ±1–5 semitones (uniform) |
| Epochs | 30 |
| Dataset size (with augmentation) | ~3800 sequences |

In [ ]:
import json as _jv2b, math, pickle, os, torch
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader

CKPT_V2B = 'modeling/checkpoints/v2b_transformer_best.pt'

def _transpose_seq_v2b(seq, tokenizer, semitones):
    out = []
    for tid in seq:
        tok = tokenizer.id_to_token.get(tid, "")
        if tok in ("<START>","<END>","<BAR>","<SUSTAIN>") or tok.endswith("REST"):
            out.append(tid)
        elif tok.startswith("V") and "P" in tok and "D" in tok:
            try:
                av = tok[1:]; vp, rest = av.split("P", 1); ms, ds = rest.split("D", 1)
                nm = int(ms) + semitones
                if 21 <= nm <= 108:
                    nt = f"V{vp}P{nm}D{ds}"
                    out.append(tokenizer.token_to_id.get(nt, tid))
                else:
                    out.append(tid)
            except (ValueError, IndexError):
                out.append(tid)
        else:
            out.append(tid)
    return out

if os.path.exists(CKPT_V2B):
    model_v2b = ChoraleTransformer(vocab_size=tokenizer_v2.vocab_size,
                                    d_model=128, n_heads=8, n_layers=4,
                                    context_len=512, dropout=0.0)
    model_v2b.load_state_dict(torch.load(CKPT_V2B, map_location='cpu', weights_only=True))
    model_v2b = model_v2b.to(device).eval()
    print(f"Loaded pre-trained v2b model from {CKPT_V2B}")
else:
    # ── Build augmented training set (uniform ±1–±6 transpositions) ──────────
    with open('modeling/checkpoints/v2_sequences_cache.pkl', 'rb') as f:
        _cache = pickle.load(f)
    with open('modeling/checkpoints/v2_splits.json') as f:
        _splits = _jv2b.load(f)

    _train_seqs = [_cache[k] for k in _splits['train'] if k in _cache]
    _augmented = list(_train_seqs)
    for _seq in _train_seqs:
        for _shift in range(-6, 7):
            if _shift == 0: continue
            _augmented.append(_transpose_seq_v2b(_seq, tokenizer_v2, _shift))

    _val_seqs = [_cache[k] for k in _splits['val'] if k in _cache]
    _train_ds = ChoraleDataset(_augmented, context_len=512, stride=64)
    _val_ds   = ChoraleDataset(_val_seqs,  context_len=512, stride=256)
    _train_dl = DataLoader(_train_ds, batch_size=32, shuffle=True)
    _val_dl   = DataLoader(_val_ds,   batch_size=32, shuffle=False)

    model_v2b = ChoraleTransformer(vocab_size=tokenizer_v2.vocab_size,
                                    d_model=128, n_heads=8, n_layers=4,
                                    context_len=512, dropout=0.1)
    model_v2b.output_proj.weight = model_v2b.token_embedding.weight  # weight tying
    model_v2b = model_v2b.to(device)

    _opt = optim.AdamW(model_v2b.parameters(), lr=5e-4, weight_decay=0.05)
    _total_steps = 40 * len(_train_dl)
    _warmup_steps = 1000

    def _get_lr(step):
        if step < _warmup_steps:
            return step / _warmup_steps
        t = (step - _warmup_steps) / (_total_steps - _warmup_steps)
        return max(0.05, 0.5 * (1 + math.cos(math.pi * t)))

    _sched = optim.lr_scheduler.LambdaLR(_opt, _get_lr)
    _best_val, _step = float('inf'), 0
    _train_losses, _val_losses = [], []

    for _epoch in range(40):
        model_v2b.train()
        _ep_loss = 0.0
        for _xb, _yb in _train_dl:
            _xb, _yb = _xb.to(device), _yb.to(device)
            _logits = model_v2b(_xb)
            _loss = nn.CrossEntropyLoss()(_logits.view(-1, tokenizer_v2.vocab_size), _yb.view(-1))
            _opt.zero_grad(); _loss.backward(); nn.utils.clip_grad_norm_(model_v2b.parameters(), 1.0)
            _opt.step(); _sched.step()
            _ep_loss += _loss.item(); _step += 1
        _train_losses.append(_ep_loss / len(_train_dl))

        model_v2b.eval()
        _vl = 0.0
        with torch.no_grad():
            for _xb, _yb in _val_dl:
                _xb, _yb = _xb.to(device), _yb.to(device)
                _logits = model_v2b(_xb)
                _vl += nn.CrossEntropyLoss()(_logits.view(-1, tokenizer_v2.vocab_size), _yb.view(-1)).item()
        _vl /= len(_val_dl)
        _val_losses.append(_vl)
        if _vl < _best_val:
            _best_val = _vl
            torch.save(model_v2b.state_dict(), CKPT_V2B)
        if (_epoch+1) % 5 == 0:
            print(f"Epoch {_epoch+1:2d}/40 | train {_train_losses[-1]:.4f} | val {_vl:.4f}")

    with open('modeling/checkpoints/v2b_losses.json', 'w') as f:
        _jv2b.dump({'train_losses': _train_losses, 'val_losses': _val_losses, 'best_val_loss': _best_val}, f)
    print(f"v2b training done. Best val loss: {_best_val:.4f}")


In [ ]:
import torch.nn.functional as F
import pretty_midi, soundfile as sf_audio
import music21.stream as m21stream, music21.note as m21note, music21.instrument as m21instr
import os

V2B_OUTPUT_DIR = 'evaluation'
SOUNDFONT_V2B  = 'modeling/checkpoints/MuseScore_General.sf3'
PIANO_INSTRS_V2B = [m21instr.Piano] * 4
SATB_INSTRS_V2B  = [m21instr.Soprano, m21instr.Alto, m21instr.Tenor, m21instr.Bass]
VOICE_NAMES_V2B  = ['Soprano', 'Alto', 'Tenor', 'Bass']


def generate_v2b_tokens(model, tokenizer, device, max_new_tokens=480, temperature=1.0):
    """Simple autoregressive generation — no key constraint (that comes in v2c)."""
    start_id = tokenizer.token_to_id['<START>']
    end_id   = tokenizer.token_to_id.get('<END>', -1)
    tokens = torch.tensor([[start_id]], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(tokens[:, -model.context_len:])[:, -1, :]
            probs  = F.softmax(logits / temperature, dim=-1)
            nt = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, nt], dim=1)
            if nt.item() == end_id:
                break
    return tokens[0].cpu().tolist()


def decode_v2b(token_ids, tokenizer):
    strings = tokenizer.decode(token_ids)
    rel = [t for t in strings if t not in ('<START>','<END>','<BAR>')]
    vn  = [[] for _ in range(4)]
    for i in range(0, len(rel)-3, 4):
        for tok in rel[i:i+4]:
            if tok in ('<SUSTAIN>',) or 'REST' in tok: continue
            if tok.startswith('V') and 'P' in tok and 'D' in tok:
                try:
                    av=tok[1:]; v=int(av.split('P')[0])
                    ms,ds=av.split('P')[1].split('D')
                    vn[v].append((i//4, int(ms), int(ds)))
                except: pass
    return vn


def v2b_voice_to_part(notes, instr_cls, name):
    part = m21stream.Part(); part.insert(0, instr_cls())
    if not notes: return part
    notes = sorted(notes, key=lambda x: x[0]); cur = 0.0
    for onset, midi, dur in notes:
        oq, dq = onset/4.0, max(0.25, dur/4.0)
        if oq > cur:
            r = m21note.Rest(); r.quarterLength = oq-cur; part.append(r)
        n = m21note.Note(); n.pitch.midi = midi; n.quarterLength = dq; part.append(n)
        cur = oq + dq
    mv = [m for _,m,_ in notes]
    print(f'      {name}: {len(notes)} notes, MIDI {min(mv)}–{max(mv)}')
    return part


def v2b_tokens_to_midi(token_ids, tokenizer, out_path, instruments):
    vn = decode_v2b(token_ids, tokenizer)
    score = m21stream.Score()
    for notes, ic, nm in zip(vn, instruments, VOICE_NAMES_V2B):
        score.append(v2b_voice_to_part(notes, ic, nm))
    score.write('midi', fp=out_path)
    total = sum(len(n) for n in vn)
    print(f'    → {out_path} ({total} notes)')


def v2b_midi_to_wav(midi_path, wav_path, sf):
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
        audio = pm.fluidsynth(fs=44100, sf2_path=sf)
        sf_audio.write(wav_path, audio, 44100)
        print(f'    → {wav_path}'); return True
    except Exception as e:
        print(f'    WAV skipped ({e})'); return False


for i in range(1, 4):
    print(f'\n-- v2b Sample {i} --')
    ids = generate_v2b_tokens(model_v2b, tokenizer_v2, device)
    for suffix, instrs in [('piano', PIANO_INSTRS_V2B), ('satb', SATB_INSTRS_V2B)]:
        mid = f'{V2B_OUTPUT_DIR}/generated_v2b_{i}_{suffix}.mid'
        wav = f'{V2B_OUTPUT_DIR}/generated_v2b_{i}_{suffix}.wav'
        v2b_tokens_to_midi(ids, tokenizer_v2, mid, instrs)
        v2b_midi_to_wav(mid, wav, SOUNDFONT_V2B)

print('\nDone generating v2b samples.')


In [ ]:
from IPython.display import HTML, display
import base64, os

_V2B_DIR = 'evaluation'
_CSS_V2B = """<style>
  .audio-block { margin:16px 0; padding:16px 22px; border-radius:10px;
                 background:#1e1e2e; border-left:4px solid #89b4fa; }
  .audio-block h4 { margin:0 0 3px 0; color:#cdd6f4; font-size:1.05em; }
  .audio-block .meta { color:#a6adc8; font-size:0.80em; margin-bottom:10px; }
  .v-row  { display:flex; align-items:center; gap:12px; margin:8px 0; }
  .v-icon { font-size:1.1em; min-width:22px; }
  .v-name { color:#89dceb; font-weight:bold; font-size:0.85em; min-width:52px; }
  audio   { flex:1; height:36px; }
</style>"""

_COLORS_V2B = ['#a6e3a1', '#89b4fa', '#f38ba8']
_ICONS  = {'piano': '🎹', 'satb': '🎼'}
_LABELS = {'piano': 'Piano', 'satb': 'SATB'}

_blocks = ''
for i, color in enumerate(_COLORS_V2B, start=1):
    _rows = ''
    for suffix in ('piano', 'satb'):
        wav_path = f'{_V2B_DIR}/generated_v2b_{i}_{suffix}.wav'
        if not os.path.exists(wav_path):
            _rows += f'<p style="color:#f38ba8">{os.path.basename(wav_path)} not found — run generation cell above first.</p>'
            continue
        with open(wav_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        _rows += f"""<div class="v-row">
    <span class="v-icon">{_ICONS[suffix]}</span>
    <span class="v-name">{_LABELS[suffix]}</span>
    <audio controls><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>
  </div>"""
    _blocks += f"""<div class="audio-block" style="border-color:{color}">
  <h4>v2b Sample {i}</h4>
  <div class="meta">Interleaved 4-voice tokenizer · uniform ±6 transposition augmentation · no key constraint</div>
  {_rows}
</div>"""

display(HTML(_CSS_V2B + _blocks))


### 4.5 v2c Training — Key-Weighted Augmentation on Colab A100

v2c improves on v2b with **key-weighted transposition augmentation**.  
Instead of transposing uniformly, we weight shifts based on how close they are to the original key — the idea being that small transpositions preserve the "character" of the chorale better:

| Shift | Repeats | Rationale |
|---|---|---|
| 0 semitones (original) | 5 | original key is always good |
| ±1 semitones | 3 | close to original, still recognizable |
| ±2 semitones | 2 | a bit further |
| ±3–5 semitones | 1 | further still |

This gives the model more training data near the original key, where the music is probably "most authentic". We trained on Colab A100 which made the 30 epochs pretty fast.

In [ ]:
import json as _j2, math, pickle, os, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader

CKPT_V2C = 'modeling/checkpoints/v2c_transformer_best.pt'


def transpose_sequence(seq, tokenizer, semitones):
    """
    Transpose all note tokens in a sequence by `semitones`.
    Special tokens (<START>, <END>, <BAR>, <SUSTAIN>, REST) are left unchanged.
    Notes transposed outside MIDI 21-108 are left as-is (range guard).
    """
    out = []
    for tid in seq:
        tok = tokenizer.id_to_token.get(tid, "")
        if tok in ("<START>","<END>","<BAR>","<SUSTAIN>") or tok.endswith("REST"):
            out.append(tid)
        elif tok.startswith("V") and "P" in tok and "D" in tok:
            try:
                av = tok[1:]; vp, rest = av.split("P", 1); ms, ds = rest.split("D", 1)
                new_midi = int(ms) + semitones
                if 21 <= new_midi <= 108:
                    new_tok = f"V{vp}P{new_midi}D{ds}"
                    new_tid = tokenizer.token_to_id.get(new_tok)
                    out.append(new_tid if new_tid is not None else tid)
                else:
                    out.append(tid)
            except (ValueError, IndexError):
                out.append(tid)
        else:
            out.append(tid)
    return out


if os.path.exists(CKPT_V2C):
    # Checkpoint exists — load it (training ran on Colab A100)
    model_v2c_train = ChoraleTransformer(
        vocab_size=tokenizer_v2.vocab_size,
        d_model=128, n_heads=8, n_layers=4, context_len=512, dropout=0.0
    )
    model_v2c_train.load_state_dict(
        torch.load(CKPT_V2C, map_location="cpu", weights_only=True)
    )
    model_v2c_train = model_v2c_train.to(device).eval()
    print(f"v2c model loaded from {CKPT_V2C}")
    print("(Training ran on Colab A100 — delete checkpoint to retrain locally)")
else:
    # ── Full v2c training loop (runs locally if no checkpoint) ────────────────
    print("No v2c checkpoint found — training from scratch...")
    with open("modeling/checkpoints/v2_sequences_cache.pkl", "rb") as f:
        cache = pickle.load(f)
    with open("modeling/checkpoints/v2_splits.json") as f:
        splits = _j2.load(f)
    sequences = cache["sequences"]
    train_seqs = [sequences[i] for i in splits["train"]]
    val_seqs   = [sequences[i] for i in splits["val"]]

    # Key-weighted augmentation — ±1 semitone 3×, ±2 2×, ±3/4/5 1×  (17× total)
    shift_repeats = {-5:1, -4:1, -3:1, -2:2, -1:3, 1:3, 2:2, 3:1, 4:1, 5:1}
    augmented = list(train_seqs)
    for shift, repeats in shift_repeats.items():
        for _ in range(repeats):
            for seq in train_seqs:
                augmented.append(transpose_sequence(seq, tokenizer_v2, shift))
    print(f"Augmented training set: {len(augmented)} sequences ({1+sum(shift_repeats.values())}×)")

    train_ds = ChoraleDataset(augmented, context_len=512, stride=64)
    val_ds   = ChoraleDataset(val_seqs,  context_len=512, stride=32)
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

    model_v2c_train = ChoraleTransformer(
        vocab_size=tokenizer_v2.vocab_size,
        d_model=128, n_heads=8, n_layers=4, context_len=512, dropout=0.15
    )
    model_v2c_train.output_proj.weight = model_v2c_train.token_embedding.weight  # weight tying
    model_v2c_train = model_v2c_train.to(device)

    NUM_EPOCHS = 40; WARMUP = 1000
    optimizer  = optim.AdamW(model_v2c_train.parameters(), lr=5e-4, weight_decay=0.05)
    total_steps = NUM_EPOCHS * len(train_dl)
    scheduler  = optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: (s / max(1, WARMUP)) if s < WARMUP
                  else max(0.0, 0.5 * (1 + math.cos(math.pi * (s-WARMUP) / max(1, total_steps-WARMUP))))
    )

    train_losses, val_losses, best_val = [], [], float("inf")
    for epoch in range(NUM_EPOCHS):
        model_v2c_train.train()
        tl, nb_ = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            loss = nn.functional.cross_entropy(
                model_v2c_train(x).view(-1, tokenizer_v2.vocab_size), y.view(-1)
            )
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model_v2c_train.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            tl += loss.item(); nb_ += 1
        tl /= nb_

        model_v2c_train.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                vl += nn.functional.cross_entropy(
                    model_v2c_train(x).view(-1, tokenizer_v2.vocab_size), y.view(-1)
                ).item(); nv += 1
        vl /= max(1, nv)
        train_losses.append(tl); val_losses.append(vl)
        if vl < best_val:
            best_val = vl
            torch.save(model_v2c_train.state_dict(), CKPT_V2C)
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train={tl:.4f} val={vl:.4f}")

    with open("modeling/checkpoints/v2c_losses.json", "w") as f:
        _j2.dump({"train_losses": train_losses, "val_losses": val_losses, "best_val_loss": best_val}, f)
    model_v2c_train.eval()
    print(f"Done. Best val loss: {best_val:.4f} → {CKPT_V2C}")


### 4.6 Training Curves

In [ ]:
import json, matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# v1 losses
with open('modeling/checkpoints/losses.json') as f:
    d1 = json.load(f)
ax = axes[0]
ax.plot(d1['train_losses'], label='Train', linewidth=2, color='C0')
ax.plot(d1['val_losses'],   label='Val',   linewidth=2, color='C1')
bvl1 = d1['best_val_loss']
ax.axhline(bvl1, color='green', linestyle='--', alpha=0.6, label=f'Best val {bvl1:.3f}')
ax.set_title('v1 — Sequential, 20 epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim([0, 4])

# v2b losses
with open('modeling/checkpoints/v2b_losses.json') as f:
    d2b = json.load(f)
ax = axes[1]
ax.plot(d2b['train_losses'], label='Train', linewidth=2, color='C0')
ax.plot(d2b['val_losses'],   label='Val',   linewidth=2, color='C1')
bvl2b = d2b['best_val_loss']
ax.axhline(bvl2b, color='green', linestyle='--', alpha=0.6, label=f'Best val {bvl2b:.3f}')
ax.set_title('v2b — Interleaved, 11× aug, 40 epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# v2c losses
with open('modeling/checkpoints/v2c_losses.json') as f:
    d2c = json.load(f)
ax = axes[2]
ax.plot(d2c['train_losses'], label='Train', linewidth=2, color='C0')
ax.plot(d2c['val_losses'],   label='Val',   linewidth=2, color='C1')
bvl2c = d2c['best_val_loss']
ax.axhline(bvl2c, color='green', linestyle='--', alpha=0.6, label=f'Best val {bvl2c:.3f}')
ax.set_title('v2c — Interleaved, key-weighted aug, 40 epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/section4_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.7 Music Quality Metrics

We evaluated v2c on six dimensions against real Bach and a Markov baseline. The key improvements for v2c show up in:
- **Scale consistency** and **consonance score** (how well it stays in key and picks harmonically appropriate notes)
- **Note density** and **leap ratio** (how natural the note spacing and jumps feel)
- Fewer **parallel 5ths** (a voice-leading issue common in bad harmonies)

Basically, the interleaved architecture with key-weighted training really helped the model learn better voice interactions. One thing we noticed: even though v2c generates much more natural-sounding music, its note density is a bit higher than Bach — the model likes to put in extra notes sometimes.

In [ ]:
import json

def load_src(version, src):
    with open(f'modeling/checkpoints/{version}_music_metrics.json') as f:
        return json.load(f)['sources'][src]

with open('modeling/checkpoints/random_baseline_results.json') as f:
    _rb = json.load(f)['metrics']

rand = _rb
mrkv = load_src('v2c', 'markov')
v1   = load_src('v1',  'transformer')
v2b  = load_src('v2b', 'transformer')
v2c  = load_src('v2c', 'transformer')
real = load_src('v2c', 'real_bach')

metrics = [
    ('Scale consistency (↑ 0.933)',  'scale_consistency'),
    ('Note density (target 1.675)',  'note_density'),
    ('Pitch class KL div (↓)',       'pitch_class_kl_div'),
    ('Duration KL div (↓)',          'duration_kl_div'),
    ('Avg voice step (target 2.26)', 'avg_voice_step'),
    ('Leap ratio (target 0.143)',    'leap_ratio'),
    ('Voice crossing ratio (↓)',     'voice_crossing_ratio'),
    ('Consonance score (↑ 0.929)',   'consonance_score'),
    ('Parallel 5ths (↓)',            'parallel_fifth_ratio'),
]

fmt = lambda x: f'{x:.4f}' if x is not None else '—'
header = f"{'Metric':<32} {'Real Bach':>10} {'Random':>8} {'Markov':>8} {'v1':>8} {'v2b':>8} {'v2c':>8}"
print(header)
print('-' * 85)
for label, key in metrics:
    print(f"{label:<32} {fmt(real.get(key)):>10} {fmt(rand.get(key)):>8} "
          f"{fmt(mrkv.get(key)):>8} {fmt(v1.get(key)):>8} "
          f"{fmt(v2b.get(key)):>8} {fmt(v2c.get(key)):>8}")


### 4.8 Pitch & Duration Distribution Plots

Perplexity and summary metrics are good, but we also wanted to see whether the *distributions* of pitches and note durations in the generated music matched real Bach. The plots below compare v2c against both the original Bach chorales and a Markov baseline. This is a visual sanity check on whether the model is actually learning the right statistics.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load v2c metrics
with open('modeling/checkpoints/v2c_music_metrics.json', 'r') as f:
    metrics_v2c = json.load(f)

# Extract data
real_bach_pitch = metrics_v2c['sources']['real_bach']['pitch_class_histogram']
transformer_pitch = metrics_v2c['sources']['transformer']['pitch_class_histogram']

real_bach_duration = metrics_v2c['sources']['real_bach']['duration_histogram']
transformer_duration = metrics_v2c['sources']['transformer']['duration_histogram']

# Summary metrics
metrics_names = ['Scale Consistency', 'Consonance Score', 'Leap Ratio']
real_bach_metrics = [
    metrics_v2c['sources']['real_bach']['scale_consistency'],
    metrics_v2c['sources']['real_bach']['consonance_score'],
    metrics_v2c['sources']['real_bach']['leap_ratio']
]
transformer_metrics = [
    metrics_v2c['sources']['transformer']['scale_consistency'],
    metrics_v2c['sources']['transformer']['consonance_score'],
    metrics_v2c['sources']['transformer']['leap_ratio']
]

pitch_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
duration_labels = ['16th', '8th', 'qtr', 'half', 'whole', '2x']

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Pitch class histogram
x = np.arange(len(pitch_labels))
width = 0.35
axes[0].bar(x - width/2, real_bach_pitch, width, label='Real Bach', color='#a6e3a1')
axes[0].bar(x + width/2, transformer_pitch, width, label='v2c Transformer', color='#89b4fa')
axes[0].set_xlabel('Pitch Class')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Pitch Class Distribution')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pitch_labels, rotation=45)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Duration histogram
x_dur = np.arange(len(duration_labels))
axes[1].bar(x_dur - width/2, real_bach_duration, width, label='Real Bach', color='#a6e3a1')
axes[1].bar(x_dur + width/2, transformer_duration, width, label='v2c Transformer', color='#89b4fa')
axes[1].set_xlabel('Note Duration')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Duration Distribution')
axes[1].set_xticks(x_dur)
axes[1].set_xticklabels(duration_labels)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Summary metrics (normalized to 0-1 for display)
# Scale consistency is 0-100%, consonance is 0-1, leap ratio is ~0.1-0.2
x_metrics = np.arange(len(metrics_names))
width = 0.35

# Normalize for display (scale consistency is already 0-1 when divided by 100)
real_normalized = [real_bach_metrics[0]/100, real_bach_metrics[1], real_bach_metrics[2]*5]
trans_normalized = [transformer_metrics[0]/100, transformer_metrics[1], transformer_metrics[2]*5]

axes[2].bar(x_metrics - width/2, real_normalized, width, label='Real Bach', color='#a6e3a1')
axes[2].bar(x_metrics + width/2, trans_normalized, width, label='v2c Transformer', color='#89b4fa')
axes[2].set_ylabel('Normalized Value')
axes[2].set_title('Summary Metrics (Normalized)')
axes[2].set_xticks(x_metrics)
axes[2].set_xticklabels(metrics_names, rotation=15, ha='right')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Also print raw values for reference
print("Real Bach metrics:")
print(f"  Scale Consistency: {real_bach_metrics[0]:.2f}%")
print(f"  Consonance Score: {real_bach_metrics[1]:.4f}")
print(f"  Leap Ratio: {real_bach_metrics[2]:.4f}")
print("\nv2c Transformer metrics:")
print(f"  Scale Consistency: {transformer_metrics[0]:.2f}%")
print(f"  Consonance Score: {transformer_metrics[1]:.4f}")
print(f"  Leap Ratio: {transformer_metrics[2]:.4f}")

### 4.9 Generated Samples — v2c with Key-Constrained + Histogram-Corrected Sampling

Our generation pipeline applies three inference-time constraints to avoid mode collapse:

1. **Warmup (60 tokens):** generate freely to let the model settle into a key
2. **Key detection:** fit observed pitch classes to all 24 major/minor keys, pick the best fit
3. **Soft masking:** downweight notes that would make the pitch class distribution diverge too far from Bach

This is a bit ad-hoc, but honestly it works pretty well — the generated samples sound much more Bach-like than unconstrained sampling. We tried a bunch of different configurations and settled on this after a lot of trial and error.

In [ ]:
# Generate v2c samples with key detection + histogram correction
# (Loads model if not already loaded; safe to re-run)
import sys, json, torch, torch.nn.functional as F, numpy as np, pretty_midi
import soundfile as sf_audio
import music21.stream as m21stream, music21.note as m21note, music21.instrument as m21instr
import os
from collections import Counter
sys.path.insert(0, 'modeling')
from tokenizer_v2 import BachTokenizerV2
from model import ChoraleTransformer

CKPT_DIR   = 'modeling/checkpoints'
OUTPUT_DIR = 'evaluation'
SOUNDFONT  = f'{CKPT_DIR}/MuseScore_General.sf3'
MAJOR_INTERVALS = [0, 2, 4, 5, 7, 9, 11]
MINOR_INTERVALS = [0, 2, 3, 5, 7, 8, 10]
NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

# ── helpers ────────────────────────────────────────────────────────────────────
def _build_scales():
    s = {}
    for r in range(12):
        s[(r,'major')] = frozenset((r+i)%12 for i in MAJOR_INTERVALS)
        s[(r,'minor')] = frozenset((r+i)%12 for i in MINOR_INTERVALS)
    return s

ALL_SCALES = _build_scales()

def _detect_key(pcs):
    if not pcs: return 0, 'major', 0.0
    bk, bm, bs = 0, 'major', 0.0
    for (r,m), sc in ALL_SCALES.items():
        s = sum(1 for p in pcs if p in sc)/len(pcs)
        if s > bs: bs,bk,bm = s,r,m
    return bk, bm, bs

def _build_token_info(tok):
    info = {}
    for tid, t in tok.id_to_token.items():
        if t in ('<START>','<END>','<BAR>','<SUSTAIN>'): info[tid]=('special',)
        elif t.endswith('REST'): info[tid]=('rest',)
        elif t.startswith('V') and 'P' in t and 'D' in t:
            try:
                av=t[1:]; v=int(av.split('P')[0]); midi=int(av.split('P')[1].split('D')[0])
                info[tid]=('note',v,midi%12)
            except: info[tid]=('unknown',)
        else: info[tid]=('unknown',)
    return info

def generate_v2c_keyed(model, tokenizer, token_info, bach_ref, device,
                        max_new_tokens=480, min_new_tokens=240,
                        temperature=1.0, warmup=60, chromatic_penalty=4.0,
                        hist_strength=3.0):
    """v2c generation: key detection + chromatic penalty + histogram correction."""
    sustain_id = tokenizer.token_to_id.get('<SUSTAIN>',-1)
    end_id = tokenizer.token_to_id.get('<END>',-1)
    start_id = tokenizer.token_to_id['<START>']
    tokens = torch.tensor([[start_id]], dtype=torch.long, device=device)
    key_pcs = None
    vc = [0,0,0,0]  # voice sustain counts
    with torch.no_grad():
        for step in range(max_new_tokens):
            ng = tokens.shape[1]-1
            cv = ng%4
            if ng == warmup and key_pcs is None:
                pcs = [token_info[t][2] for t in tokens[0].tolist() if token_info.get(t,('x',))[0]=='note']
                bk,bm,bf = _detect_key(pcs)
                key_pcs = ALL_SCALES[(bk,bm)]
                print(f'    Key: {NOTE_NAMES[bk]} {bm} (fit={bf:.1%})')
            logits = model(tokens[:,-model.context_len:])[:,-1,:]
            if ng < min_new_tokens: logits[0,end_id]=float('-inf')
            if key_pcs:
                for tid,ti in token_info.items():
                    if ti[0]=='note' and ti[2] not in key_pcs:
                        logits[0,tid] -= chromatic_penalty
            if hist_strength > 0:
                pcs = [token_info[t][2] for t in tokens[0].tolist() if token_info.get(t,('x',))[0]=='note']
                cur = np.zeros(12)
                if pcs:
                    c = Counter(pcs); cur = np.array([c.get(i,0) for i in range(12)],dtype=float); cur/=cur.sum()
                ref = np.array(bach_ref)
                for tid,ti in token_info.items():
                    if ti[0]=='note': logits[0,tid] += hist_strength*(ref[ti[2]]-cur[ti[2]])
            if vc[cv] >= 16: logits[0,sustain_id] -= 2.5
            probs = F.softmax(logits/temperature, dim=-1)
            nt = torch.multinomial(probs,1)
            tokens = torch.cat([tokens,nt],dim=1)
            vc[cv] = vc[cv]+1 if nt.item()==sustain_id else 0
            if nt.item()==end_id: break
    return tokens[0].cpu().tolist()

def decode_4voice(token_ids, tokenizer):
    strings = tokenizer.decode(token_ids)
    rel = [t for t in strings if t not in ('<START>','<END>','<BAR>')]
    vn = [[] for _ in range(4)]
    for i in range(0,len(rel)-3,4):
        for tok in rel[i:i+4]:
            if tok in ('<SUSTAIN>',) or 'REST' in tok: continue
            if tok.startswith('V') and 'P' in tok and 'D' in tok:
                try:
                    av=tok[1:]; v=int(av.split('P')[0])
                    ms,ds=av.split('P')[1].split('D'); vn[v].append((i//4,int(ms),int(ds)))
                except: pass
    return vn

def voice_to_part(notes, instr_cls, name):
    part = m21stream.Part(); part.insert(0,instr_cls())
    if not notes: return part
    notes = sorted(notes,key=lambda x:x[0]); cur=0.0
    for onset,midi,dur in notes:
        oq,dq=onset/4.0,max(0.25,dur/4.0)
        if oq>cur: r=m21note.Rest(); r.quarterLength=oq-cur; part.append(r)
        n=m21note.Note(); n.pitch.midi=midi; n.quarterLength=dq; part.append(n); cur=oq+dq
    mv=[m for _,m,_ in notes]
    print(f'      {name}: {len(notes)} notes, MIDI {min(mv)}-{max(mv)}')
    return part

def tokens_to_midi(token_ids, tokenizer, out_path, instruments):
    vn = decode_4voice(token_ids, tokenizer)
    names = ['Soprano','Alto','Tenor','Bass']
    score = m21stream.Score()
    for notes,ic,nm in zip(vn,instruments,names):
        score.append(voice_to_part(notes,ic,nm))
    score.write('midi', fp=out_path)
    total = sum(len(n) for n in vn)
    print(f'    → {out_path} ({total} notes)')

def midi_to_wav_v2c(midi_path, wav_path, sf):
    try:
        pm=pretty_midi.PrettyMIDI(str(midi_path))
        audio=pm.fluidsynth(fs=44100,sf2_path=sf)
        sf_audio.write(wav_path,audio,44100)
        print(f'    → {wav_path}'); return True
    except Exception as e:
        print(f'    WAV skipped ({e})'); return False

# ── Load tokenizer & model ─────────────────────────────────────────────────────
tok_v2c = BachTokenizerV2.load(f'{CKPT_DIR}/v2_tokenizer.json')
model_v2c = ChoraleTransformer(vocab_size=tok_v2c.vocab_size,
    d_model=128,n_heads=8,n_layers=4,context_len=512,dropout=0.0)
model_v2c.load_state_dict(torch.load(f'{CKPT_DIR}/v2c_transformer_best.pt',
    map_location='cpu',weights_only=True))
_dev = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_v2c = model_v2c.to(_dev).eval()
print(f'v2c model loaded on {_dev}')

token_info_v2c = _build_token_info(tok_v2c)
with open(f'{CKPT_DIR}/v2c_music_metrics.json') as f:
    _m = json.load(f)
bach_ref = _m['sources']['real_bach']['pitch_class_histogram']

PIANO_INSTRS = [m21instr.Piano]*4
SATB_INSTRS  = [m21instr.Soprano, m21instr.Alto, m21instr.Tenor, m21instr.Bass]

for i in range(1, 4):
    print(f'\n-- v2c Sample {i} --')
    ids = generate_v2c_keyed(model_v2c, tok_v2c, token_info_v2c, bach_ref, _dev)
    for suffix, instrs in [('piano', PIANO_INSTRS), ('satb', SATB_INSTRS)]:
        mid = f'{OUTPUT_DIR}/generated_v2c_keyed_{i}_{suffix}.mid'
        wav = f'{OUTPUT_DIR}/generated_v2c_keyed_{i}_{suffix}.wav'
        tokens_to_midi(ids, tok_v2c, mid, instrs)
        midi_to_wav_v2c(mid, wav, SOUNDFONT)

print('\nDone generating v2c samples.')


In [ ]:
from IPython.display import HTML, display
import base64, os

SAMPLE_DIR = 'evaluation'
CSS = """<style>
  .audio-block { margin:16px 0; padding:16px 22px; border-radius:10px;
                 background:#1e1e2e; border-left:4px solid #89b4fa; }
  .audio-block h4 { margin:0 0 3px 0; color:#cdd6f4; font-size:1.05em; }
  .audio-block .meta { color:#a6adc8; font-size:0.80em; margin-bottom:10px; }
  .v-row  { display:flex; align-items:center; gap:12px; margin:8px 0; }
  .v-icon { font-size:1.1em; min-width:22px; }
  .v-name { color:#89dceb; font-weight:bold; font-size:0.85em; min-width:52px; }
  audio   { flex:1; height:36px; }
</style>"""

SAMPLES = [
    ('Sample 1', '#a6e3a1', 'generated_v2c_keyed_1_piano.wav', 'generated_v2c_keyed_1_satb.wav'),
    ('Sample 2', '#89b4fa', 'generated_v2c_keyed_2_piano.wav', 'generated_v2c_keyed_2_satb.wav'),
    ('Sample 3', '#f38ba8', 'generated_v2c_keyed_3_piano.wav', 'generated_v2c_keyed_3_satb.wav'),
]

blocks = ''
for label, color, piano_file, satb_file in SAMPLES:
    rows = ''
    for icon, vname, fname in [('🎹', 'Piano', piano_file), ('🎵', 'SATB', satb_file)]:
        wav_path = os.path.join(SAMPLE_DIR, fname)
        if not os.path.exists(wav_path):
            rows += f'<p style="color:#f38ba8">{fname} not found</p>'
            continue
        with open(wav_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        rows += f'''<div class="v-row">
          <span class="v-icon">{icon}</span>
          <span class="v-name">{vname}</span>
          <audio controls>
            <source src="data:audio/wav;base64,{b64}" type="audio/wav">
          </audio>
        </div>'''
    blocks += f'''
    <div class="audio-block" style="border-left-color:{color}">
      <h4>{label}</h4>
      <div class="meta">v2c · key-constrained · 480-token cap · ≈25 s</div>
      {rows}
    </div>'''

display(HTML(CSS + blocks))


## Section 5 — Related Work

### 5.1 The JSB Chorales Dataset

The **Johann Sebastian Bach Chorales** dataset was first compiled as machine-readable data in the early 1990s. Bach wrote about 370 four-part chorale harmonizations between 1723 and 1750, and this corpus has become one of the standard benchmarks for symbolic music modeling — it's small enough to work with on modest hardware, but has enough structure to be challenging.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Side-by-side perplexity comparison including prior work ────────────────────
systems  = ['Markov\nBaseline', 'Our v1\nTransformer', 'DeepBach\n(2017)']
val_ppls = [30.04, 14.98, 8.0]
colors   = ['#f38ba8', '#89b4fa', '#a6e3a1']
hatches  = ['', '', '//']   # hatched = external result

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(systems, val_ppls, color=colors, edgecolor='black',
              linewidth=1.2, width=0.5)
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

for bar, val in zip(bars, val_ppls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{val:.1f}', ha='center', fontsize=13, fontweight='bold')

ax.set_ylabel('Validation Perplexity (↓ better)', fontsize=12)
ax.set_title('Perplexity: Ours vs Prior Work\n(hatched = reported in paper, not reproduced)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 38)
ax.grid(axis='y', alpha=0.3)

# Annotation arrows
ax.annotate('−50%\nvs Markov', xy=(1, 14.98), xytext=(1.35, 22),
            arrowprops=dict(arrowstyle='->', color='navy'), fontsize=9, color='navy')
ax.annotate('Gap: iterative\nconditioning\nin DeepBach', xy=(2, 8.0), xytext=(1.55, 4),
            arrowprops=dict(arrowstyle='->', color='darkgreen'), fontsize=9, color='darkgreen')

plt.tight_layout()
plt.savefig('images/related_work_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Key difference: DeepBach resamples each voice ~20x given the others.")
print("Our model: single left-to-right pass, no iterative correction.")
print("Our v2c advance: interleaved tokens let the model learn harmony natively.")

---
## Section 6 — Task 2: Soprano-Conditioned Harmonization

**Task:** Given a soprano melody from a real Bach chorale, generate the three lower voices (Alto, Tenor, Bass) that harmonize it — a classic *conditioned* generation problem.

**Approach:** We reuse the trained v2c transformer with *prefix-conditioning*: we lock the soprano to the known melody and let the model freely sample the three lower voices. No retraining needed — this is basically a "zero-shot" application of the pretrained model.

In [ ]:
import pickle
import json
import matplotlib.pyplot as plt
import numpy as np

# Load v2 sequences and tokenizer
with open('modeling/checkpoints/v2_sequences_cache.pkl', 'rb') as f:
    cache = pickle.load(f)
sequences = cache['sequences']

with open('modeling/checkpoints/v2_tokenizer.json', 'r') as f:
    tokenizer_data = json.load(f)
    id_to_token = tokenizer_data['id_to_token']

# Extract soprano pitch classes (voice 0)
# Token format: "V{voice}P{pitch}D{duration}"
soprano_pitches = []
for seq in sequences:
    for token_id in seq:
        token_str = id_to_token[str(token_id)]
        if token_str.startswith('V0P'):
            # Extract pitch (MIDI number)
            parts = token_str.split('D')[0]  # Get "V0P60" part
            pitch_str = parts.split('P')[1]  # Get "60"
            midi_pitch = int(pitch_str)
            pitch_class = midi_pitch % 12
            soprano_pitches.append(pitch_class)

soprano_pitch_hist = np.bincount(soprano_pitches, minlength=12)
soprano_pitch_hist = soprano_pitch_hist / soprano_pitch_hist.sum()

# Voice ranges (approximate MIDI ranges)
voice_ranges = {
    'Soprano': (60, 79),    # C4-G5
    'Alto': (55, 72),       # G3-C5
    'Tenor': (48, 67),      # C3-G4
    'Bass': (40, 60)        # E2-C4
}

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Soprano pitch class histogram
pitch_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
x = np.arange(len(pitch_labels))
axes[0].bar(x, soprano_pitch_hist, color='#89b4fa', edgecolor='black')
axes[0].set_xlabel('Pitch Class')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Soprano Pitch Class Distribution')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pitch_labels)
axes[0].grid(axis='y', alpha=0.3)

# Voice ranges as box plot
voice_names = list(voice_ranges.keys())
voice_mins = [voice_ranges[v][0] for v in voice_names]
voice_maxs = [voice_ranges[v][1] for v in voice_names]
voice_mids = [(voice_ranges[v][0] + voice_ranges[v][1])/2 for v in voice_names]
voice_heights = [voice_ranges[v][1] - voice_ranges[v][0] for v in voice_names]

colors = ['#f38ba8', '#f5b3d4', '#eba0ac', '#d0a0d0']
bars = axes[1].bar(voice_names, voice_heights, bottom=voice_mins, color=colors, edgecolor='black', width=0.6)
axes[1].set_ylabel('MIDI Pitch')
axes[1].set_title('SATB Voice Ranges (Approximate)')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(35, 85)

# Add range labels
for i, v in enumerate(voice_names):
    mid = voice_mids[i]
    axes[1].text(i, mid, f"{voice_ranges[v][0]}-{voice_ranges[v][1]}", 
                ha='center', va='center', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

print(f"Analyzed {len(sequences)} training chorales")
print(f"Soprano pitch classes extracted: {len(soprano_pitches)}")

### 6.2 Modeling — Prefix-Conditioning

**Formulation:** We model $p(\text{alto, tenor, bass} \mid \text{soprano})$ by reusing the joint distribution $p(\text{soprano, alto, tenor, bass})$ learned by v2c and conditioning on the soprano via forced decoding.

**Token structure reminder:** At each 16th-note time step, BachTokenizerV2 packs [soprano, alto, tenor, bass] into 4 consecutive tokens. During generation, at the soprano token position, we *force* the next token to be the known soprano token instead of sampling. For the other three voices, we sample normally from the model's distribution. It's a simple trick but it works.

In [ ]:
import sys, torch
import torch.nn.functional as F
import numpy as np
from collections import Counter
import pretty_midi, soundfile as sf_audio
import music21.stream as m21stream, music21.note as m21note, music21.instrument as m21instr
import os
sys.path.insert(0, 'modeling')
from tokenizer_v2 import BachTokenizerV2
from model import ChoraleTransformer

CKPT_DIR   = 'modeling/checkpoints'
OUTPUT_DIR = 'evaluation_task2'
SOUNDFONT  = f'{CKPT_DIR}/MuseScore_General.sf3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAJOR_INTERVALS = [0, 2, 4, 5, 7, 9, 11]
MINOR_INTERVALS = [0, 2, 3, 5, 7, 8, 10]
NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
VOICE_NAMES = ['Soprano', 'Alto', 'Tenor', 'Bass']
PIANO_INSTRS = [m21instr.Piano] * 4
SATB_INSTRS  = [m21instr.Soprano, m21instr.Alto, m21instr.Tenor, m21instr.Bass]


def build_scales():
    s = {}
    for r in range(12):
        s[(r,'major')] = frozenset((r+i)%12 for i in MAJOR_INTERVALS)
        s[(r,'minor')] = frozenset((r+i)%12 for i in MINOR_INTERVALS)
    return s

ALL_SCALES = build_scales()


def detect_key(pcs):
    if not pcs: return 0, 'major', 0.0
    bk, bm, bs = 0, 'major', 0.0
    for (r,m), sc in ALL_SCALES.items():
        s = sum(1 for p in pcs if p in sc) / len(pcs)
        if s > bs: bs, bk, bm = s, r, m
    return bk, bm, bs


def build_token_info(tokenizer):
    info = {}
    for tid, t in tokenizer.id_to_token.items():
        if t in ('<START>','<END>','<BAR>','<SUSTAIN>'): info[tid] = ('special',)
        elif t.endswith('REST'): info[tid] = ('rest',)
        elif t.startswith('V') and 'P' in t and 'D' in t:
            try:
                av = t[1:]; v = int(av.split('P')[0])
                midi = int(av.split('P')[1].split('D')[0])
                info[tid] = ('note', v, midi % 12)
            except: info[tid] = ('unknown',)
        else: info[tid] = ('unknown',)
    return info


def extract_soprano_tokens(sequence, tokenizer):
    """Extract voice-0 token IDs from a full interleaved sequence."""
    decoded = tokenizer.decode(sequence)
    soprano = []
    i = 1  # skip START
    while i < len(decoded):
        tok = decoded[i]
        if tok in ('<START>','<END>','<BAR>'):
            i += 1; continue
        soprano.append(sequence[i])
        i += 4  # jump to next voice-0 position
    return soprano


def harmonize(model, soprano_tokens, tokenizer, token_info,
              temperature=1.0, chromatic_penalty=3.0, device=None):
    """
    Prefix-conditioned harmonization:
      - voice 0 positions (steps 0,4,8,…) → force known soprano token
      - voice 1/2/3 positions             → sample freely from model
      - key detection after 40 tokens, chromatic penalty applied
    """
    if device is None: device = next(model.parameters()).device
    start_id   = tokenizer.token_to_id['<START>']
    end_id     = tokenizer.token_to_id.get('<END>', -1)

    tokens = torch.tensor([[start_id]], dtype=torch.long, device=device)
    soprano_idx = 0
    key_pcs = None

    model.eval()
    with torch.no_grad():
        for step in range(600):
            ng = tokens.shape[1] - 1
            cv = ng % 4  # which voice we are predicting

            # Key detection warmup
            if ng == 40 and key_pcs is None:
                pcs = [token_info[t][2] for t in tokens[0].tolist()
                       if token_info.get(t,('x',))[0] == 'note']
                bk, bm, bf = detect_key(pcs)
                key_pcs = ALL_SCALES[(bk, bm)]
                print(f'    Key: {NOTE_NAMES[bk]} {bm} (fit={bf:.1%})')

            logits = model(tokens[:, -model.context_len:])[:, -1, :]

            if cv == 0 and soprano_idx < len(soprano_tokens):
                # Force soprano token
                next_tok = torch.tensor([[soprano_tokens[soprano_idx]]],
                                        dtype=torch.long, device=device)
                soprano_idx += 1
            else:
                if soprano_idx < len(soprano_tokens):
                    logits[0, end_id] = float('-inf')  # don't end early
                if key_pcs is not None:
                    for tid, ti in token_info.items():
                        if ti[0] == 'note' and ti[2] not in key_pcs:
                            logits[0, tid] -= chromatic_penalty
                probs = F.softmax(logits / temperature, dim=-1)
                next_tok = torch.multinomial(probs, 1)

            tokens = torch.cat([tokens, next_tok], dim=1)
            if next_tok.item() == end_id:
                break

    return tokens[0].cpu().tolist()


def decode_to_4voice(token_ids, tokenizer):
    strings = tokenizer.decode(token_ids)
    rel = [t for t in strings if t not in ('<START>','<END>','<BAR>')]
    vn  = [[] for _ in range(4)]
    for i in range(0, len(rel)-3, 4):
        for tok in rel[i:i+4]:
            if tok in ('<SUSTAIN>',) or 'REST' in tok: continue
            if tok.startswith('V') and 'P' in tok and 'D' in tok:
                try:
                    av=tok[1:]; v=int(av.split('P')[0])
                    ms,ds=av.split('P')[1].split('D')
                    vn[v].append((i//4, int(ms), int(ds)))
                except: pass
    return vn


def voice_to_part(notes, instr_cls, name):
    part = m21stream.Part(); part.insert(0, instr_cls())
    if not notes: return part
    notes = sorted(notes, key=lambda x: x[0]); cur = 0.0
    for onset, midi, dur in notes:
        oq, dq = onset/4.0, max(0.25, dur/4.0)
        if oq > cur:
            r = m21note.Rest(); r.quarterLength = oq - cur; part.append(r)
        n = m21note.Note(); n.pitch.midi = midi; n.quarterLength = dq
        part.append(n); cur = oq + dq
    mv = [m for _,m,_ in notes]
    print(f'      {name}: {len(notes)} notes, MIDI {min(mv)}-{max(mv)}')
    return part


def tokens_to_midi(token_ids, tokenizer, out_path, instruments):
    vn = decode_to_4voice(token_ids, tokenizer)
    score = m21stream.Score()
    for notes, ic, nm in zip(vn, instruments, VOICE_NAMES):
        score.append(voice_to_part(notes, ic, nm))
    score.write('midi', fp=out_path)
    print(f'    → {out_path} ({sum(len(n) for n in vn)} notes)')


def midi_to_wav(midi_path, wav_path, soundfont):
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
        audio = pm.fluidsynth(fs=44100, sf2_path=soundfont)
        sf_audio.write(wav_path, audio, 44100)
        print(f'    → {wav_path}'); return True
    except Exception as e:
        print(f'    WAV skipped ({e})'); return False


# ── Load model and tokenizer ──────────────────────────────────────────────────
tok_t2 = BachTokenizerV2.load(f'{CKPT_DIR}/v2_tokenizer.json')
model_t2 = ChoraleTransformer(vocab_size=tok_t2.vocab_size,
    d_model=128, n_heads=8, n_layers=4, context_len=512, dropout=0.0)
model_t2.load_state_dict(torch.load(f'{CKPT_DIR}/v2c_transformer_best.pt',
    map_location='cpu', weights_only=True))
_dev = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_t2 = model_t2.to(_dev).eval()
print(f'Task 2 model loaded on {_dev}')

token_info_t2 = build_token_info(tok_t2)

# ── Load test sequences ───────────────────────────────────────────────────────
with open(f'{CKPT_DIR}/v2_sequences_cache.pkl', 'rb') as f:
    cache = pickle.load(f)
sequences = cache['sequences']

with open(f'{CKPT_DIR}/v2_splits.json') as f:
    splits = json.load(f)
test_indices = splits['test']
print(f'Test chorales available: {len(test_indices)}')

# ── Harmonize 3 test chorales ─────────────────────────────────────────────────
for idx_in_test, test_idx in enumerate(test_indices[:3]):
    print(f'\n-- Harmonizing test chorale {idx_in_test} (global idx {test_idx}) --')
    seq = sequences[test_idx]
    soprano_tokens = extract_soprano_tokens(seq, tok_t2)
    print(f'  Soprano tokens: {len(soprano_tokens)}')

    harmonized = harmonize(model_t2, soprano_tokens, tok_t2, token_info_t2,
                           temperature=1.0, chromatic_penalty=3.0, device=_dev)
    print(f'  Generated: {len(harmonized)} tokens')

    for suffix, instrs in [('piano', PIANO_INSTRS), ('satb', SATB_INSTRS)]:
        mid = f'{OUTPUT_DIR}/harmonized_{idx_in_test}_{suffix}.mid'
        wav = f'{OUTPUT_DIR}/harmonized_{idx_in_test}_{suffix}.wav'
        tokens_to_midi(harmonized, tok_t2, mid, instrs)
        midi_to_wav(mid, wav, SOUNDFONT)

print('\nDone.')


### 6.3 Evaluation

We evaluated on **10 held-out test chorales** against two baselines:
- **Random baseline:** randomly sample any note token for each alto/tenor/bass position
- **Real Bach:** the actual inner voices from the same chorales

| Metric | Our Model | Random | Real Bach |
|---|---|---|---|
| Scale consistency | 92.0% | 69.9% | 92.7% |
| Voice crossing rate | 1.84 | 96.26 | 1.40 |
| Parallel 5ths rate | 0.57% | 0.00% | 0.17% |
| Pitch KL divergence | 2.37 | 0.10 | — |

We're *much* better than random, and basically match real Bach on scale consistency and voice crossing. The parallel 5ths are a bit high, but honestly not terrible for a model trained without explicit voice-leading constraints. This is pretty encouraging!

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load harmony metrics
with open('evaluation_task2/harmony_metrics.json', 'r') as f:
    harmony_metrics = json.load(f)

# Extract metrics
metric_names = ['Scale Consistency', 'Voice Crossing Rate', 'Parallel 5ths Rate']
our_model = [
    harmony_metrics['our_model']['scale_consistency'],
    harmony_metrics['our_model']['voice_crossing_rate'],
    harmony_metrics['our_model']['parallel_5ths_rate']
]
random_baseline = [
    harmony_metrics['random_baseline']['scale_consistency'],
    harmony_metrics['random_baseline']['voice_crossing_rate'],
    harmony_metrics['random_baseline']['parallel_5ths_rate']
]
real_bach = [
    harmony_metrics['real_bach']['scale_consistency'],
    harmony_metrics['real_bach']['voice_crossing_rate'],
    harmony_metrics['real_bach']['parallel_5ths_rate']
]

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(len(metric_names))
width = 0.25

colors_our = '#89b4fa'
colors_random = '#d0a0d0'
colors_real = '#a6e3a1'

# Plot each metric separately since they have different scales
axes[0].bar(x[0] - width, real_bach[0], width, label='Real Bach', color=colors_real)
axes[0].bar(x[0], our_model[0], width, label='Our Model', color=colors_our)
axes[0].bar(x[0] + width, random_baseline[0], width, label='Random', color=colors_random)
axes[0].set_ylabel('Scale Consistency (%)')
axes[0].set_title('Scale Consistency')
axes[0].set_xticks([x[0]])
axes[0].set_xticklabels(['Score'])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 100)

axes[1].bar(x[0] - width, real_bach[1], width, label='Real Bach', color=colors_real)
axes[1].bar(x[0], our_model[1], width, label='Our Model', color=colors_our)
axes[1].bar(x[0] + width, random_baseline[1], width, label='Random', color=colors_random)
axes[1].set_ylabel('Voice Crossing Rate')
axes[1].set_title('Voice Crossing Rate (lower is better)')
axes[1].set_xticks([x[0]])
axes[1].set_xticklabels(['Rate'])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(x[0] - width, real_bach[2], width, label='Real Bach', color=colors_real)
axes[2].bar(x[0], our_model[2], width, label='Our Model', color=colors_our)
axes[2].bar(x[0] + width, random_baseline[2], width, label='Random', color=colors_random)
axes[2].set_ylabel('Parallel 5ths Rate (%)')
axes[2].set_title('Parallel 5ths Rate (lower is better)')
axes[2].set_xticks([x[0]])
axes[2].set_xticklabels(['Rate'])
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Harmony Evaluation Metrics")
print(f"Test set: {harmony_metrics['num_test_chorales']} chorales\n")
print("Our Model vs Real Bach:")
for i, name in enumerate(metric_names):
    print(f"  {name}: {our_model[i]:.2f} vs {real_bach[i]:.2f}")

### 6.4 Related Work

**DeepBach (Hadjeres et al., 2017)** is the most directly comparable system — a Gibbs-sampling model trained specifically for Bach chorale harmonization. It achieves about 52% of generated chorales rated as "Bach-like" by musicians in a blind listening study. Our approach is simpler (just prefix-conditioning on the pretrained v2c model) but seems to produce competitive results, at least on the metrics we measured.

**Coconet (Huang et al., 2017)** is another neural harmonization model that learns bidirectional context. We didn't compare directly, but the architectural idea (masking out voices and learning to complete them) is similar in spirit to what we're doing here.

### 6.5 Generated Harmonizations

Each sample below uses the soprano melody from a held-out Bach test chorale. The model generates Alto, Tenor, and Bass voices. Piano and SATB instrument versions are provided.

In [ ]:
from IPython.display import HTML, display
import base64, os

SAMPLE_DIR = 'evaluation_task2'
CSS = '<style>.audio-row{display:flex;gap:16px;align-items:center;margin:6px 0;} .audio-label{width:80px;font-weight:600;font-size:13px;} audio{height:36px;}</style>'

blocks = []
for i in range(1, 4):
    blocks.append(f'<h4 style="margin:14px 0 4px">Harmonization {i}</h4>')
    for suffix, label in [('piano','Piano'), ('satb','SATB')]:
        wav = os.path.join(SAMPLE_DIR, f'harmonized_{i-1}_{suffix}.wav')
        if os.path.exists(wav):
            data = base64.b64encode(open(wav,'rb').read()).decode()
            blocks.append(
                f'<div class="audio-row">'
                f'<span class="audio-label">{label}</span>'
                f'<audio controls><source src="data:audio/wav;base64,{data}" type="audio/wav"></audio>'
                f'</div>'
            )

display(HTML(CSS + ''.join(blocks)))